In [2]:
import torch                    # Tensor type으로 parsing하기 위해 로드
import torch.nn as nn           # torch 안 nn 부분만 로드하여 nn 별칭으로 사용 (nn → 기본 뼈대)
import torch.optim as optim     # optimizer (기울기의 변화를 주는 기능)

In [105]:
# 독립, 종속 데이터를 tensor로 생성

X = torch.tensor([[1.0], [2.0], [3.0], [4.0]])      # 2차원으로 생성
y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])      # sklearn에서는 1차원이었지만 torch에서는 2차원
                                                    # 독립변수에 2를 곱하고 1을 더한 값

In [106]:
type(X)

torch.Tensor

In [107]:
# 순전파 (모델 학습 → 예측)
    # class 클래스명(부모클래스): → 부모클래스의 기능을 상속받아서 클래스를 선언
class LinearReg(nn.Module):
    # torch의 모듈을 이용한 클래스 생성 시 2개의 함수를 선언 (생성자 함수, forward 함수)
    def __init__(self):
        # self : 자기 자신 (클래스를 생성할 때 저장되는 위치)
        # super() : 부모 클래스(nn.Module)
        super(LinearReg, self).__init__()
            # 부모 클래스의 생성자 함수 실행
        self.linear = nn.Linear(1, 1)
            # 선형 회귀 모델을 이용
            # nn.Linear 첫번째 인자값 : 독립변수 차원의 수
            #           두번째 인자값 : 출력 데이터 차원의 수
    
    def forward(self, x):
        return self.linear(x)

In [108]:
# 클래스 생성 → 회귀 모델을 생성
model = LinearReg()

In [109]:
# 손실 함수
criterion = nn.MSELoss()

In [110]:
# optimizer 설정 → 가중치를 update
# 첫번째 인자 → 어떤 모델의 파라미터를 설정할 것인가 지정
# lr → 경사 하강법의 보폭
optimizer = optim.SGD(model.parameters(), lr = 0.01)

In [111]:
# 순전파 (생성된 모델을 호출하면 forward() 함수를 호출하도록 nn.Module에서 설정되어있음)
pred = model(X)
# LinearReg 클래스 안의 forward 함수를 호출하여 독립변수(x)를 인자값으로 사용한다.

# 손실 함수
loss = criterion(pred, y)

# 기울기 초기화
optimizer.zero_grad()

# 역전파 (자동 미분) → 데이터가 있는 쪽으로 방향을 제시한다
loss.backward()

# 가중치를 업데이트 (파라미터(모델) 수정)
optimizer.step()

# loss 값을 확인
print(loss)

tensor(49.5290, grad_fn=<MseLossBackward0>)


In [112]:
# DL 모델은 반복 학습이 기본 설정 → 학습 모드를 평가 모드로 전환
# eval(): 모델을 평가모드로 전환
# train(): 모델을 학습 모드로 전환
model.eval()

# 예측, 평가 (메모리의 사용량을 줄이기 위해서 가중치의 계산을 잠시 비활성화)
with torch.no_grad():
    y_pred = model(X)
    loss = criterion(y_pred, y)
    print(y_pred)
    print(loss)

tensor([[0.3822],
        [0.4842],
        [0.5862],
        [0.6881]])
tensor(34.3674)


In [113]:
# 반복 학습을 통해서 가중치와 편향을 변화시킨다.
epochs = 200
model.train()

for epoch in range(epochs):
    # 순전파
    pred = model(X)
    # 손실 함수
    loss = criterion(pred, y)
    # 기울기 초기화
    optimizer.zero_grad()
    # 자동 미분(역전파) → 가중치의 방향을 제시
    loss.backward()
    # 가중치를 업데이트
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch: [{epoch+1}, 200], Loss: {round(loss.item(), 6)}")

Epoch: [20, 200], Loss: 0.033812
Epoch: [40, 200], Loss: 0.000607
Epoch: [60, 200], Loss: 0.000518
Epoch: [80, 200], Loss: 0.00046
Epoch: [100, 200], Loss: 0.000408
Epoch: [120, 200], Loss: 0.000362
Epoch: [140, 200], Loss: 0.000321
Epoch: [160, 200], Loss: 0.000285
Epoch: [180, 200], Loss: 0.000252
Epoch: [200, 200], Loss: 0.000224


In [114]:
model.eval()

with torch.no_grad():
    y_pred = model(X)
    loss = criterion(y_pred, y)
    print(y_pred)
    print(loss)

tensor([[2.9759],
        [4.9883],
        [7.0007],
        [9.0132]])
tensor(0.0002)


In [115]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import numpy as np

In [116]:
data = fetch_california_housing()

X = data['data']
y = data['target']

print(X.shape, y.shape)

(20640, 8) (20640,)


In [117]:
# 1차원 데이터를 2차원으로 변경
y = y.reshape(-1, 1)

In [118]:
# 학습, 평가 데이터로 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [119]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [120]:
y_train_tensor

tensor([[1.0300],
        [3.8210],
        [1.7260],
        ...,
        [2.2210],
        [2.8350],
        [3.2500]])

In [121]:
#선형 회귀 모델 객체를 선언
class Reg(nn.Module):
    # class 생성할 때 입력 데이터의 feature 수를 필수 인자로 설정
    def __init__(self, _dim):
        super(Reg, self).__init__()
        self.linear = nn.Linear(_dim, 1)
    
    def forward(self, x):
        return self.linear(x)

In [122]:
n_feature = X_train.shape[1]
model = Reg(n_feature)

In [123]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.001)

In [124]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_sc = torch.tensor(scaler.fit_transform(X_train_tensor), dtype=torch.float32)
X_test_sc = torch.tensor(scaler.transform(X_test_tensor), dtype=torch.float32)

In [125]:
epochs = 300

for epoch in range(epochs):
    pred = model(X_train_sc)
    loss = criterion(pred, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 30 == 0:
        print(f"Epoch: [{epoch+1}, 300], Loss: {round(loss.item(), 6)}")

Epoch: [30, 300], Loss: 5.1824
Epoch: [60, 300], Loss: 4.6617
Epoch: [90, 300], Loss: 4.20016
Epoch: [120, 300], Loss: 3.791021
Epoch: [150, 300], Loss: 3.428303
Epoch: [180, 300], Loss: 3.106712
Epoch: [210, 300], Loss: 2.821558
Epoch: [240, 300], Loss: 2.56869
Epoch: [270, 300], Loss: 2.344431
Epoch: [300, 300], Loss: 2.145524


In [126]:
model.eval()

with torch.no_grad():
    pred = model(X_test_sc)
    loss = criterion(pred, y_test_tensor)

    print(round(loss.item(), 6))

2.134394


In [127]:
for i in range(10):
    print(f'실제 데이터 : {y_test[i]}, 예측 데이터 : {pred[i].item()}')

실제 데이터 : [0.477], 예측 데이터 : 0.25146645307540894
실제 데이터 : [0.458], 예측 데이터 : 0.5939168930053711
실제 데이터 : [5.00001], 예측 데이터 : 0.8162897229194641
실제 데이터 : [2.186], 예측 데이터 : 1.4951975345611572
실제 데이터 : [2.78], 예측 데이터 : 0.9224231839179993
실제 데이터 : [1.587], 예측 데이터 : 1.1923649311065674
실제 데이터 : [1.982], 예측 데이터 : 1.1268112659454346
실제 데이터 : [1.575], 예측 데이터 : 0.7845538854598999
실제 데이터 : [3.4], 예측 데이터 : 1.5179917812347412
실제 데이터 : [4.466], 예측 데이터 : 1.8604823350906372


In [130]:
model2 = Reg(n_feature)
criterion2 = nn.MSELoss()
optimizer2 = optim.SGD(model2.parameters(), lr = 1e-07)

In [131]:
for epoch in range(300):
    pred2 = model2(X_train_tensor)
    loss2 = criterion2(pred2, y_train_tensor)
    optimizer2.zero_grad()
    loss2.backward()
    optimizer2.step()
    n = epoch + 1

    if n % 30 == 0:
        print(loss2)

tensor(235.8552, grad_fn=<MseLossBackward0>)
tensor(218.3774, grad_fn=<MseLossBackward0>)
tensor(202.2165, grad_fn=<MseLossBackward0>)
tensor(187.2731, grad_fn=<MseLossBackward0>)
tensor(173.4556, grad_fn=<MseLossBackward0>)
tensor(160.6792, grad_fn=<MseLossBackward0>)
tensor(148.8653, grad_fn=<MseLossBackward0>)
tensor(137.9414, grad_fn=<MseLossBackward0>)
tensor(127.8404, grad_fn=<MseLossBackward0>)
tensor(118.5002, grad_fn=<MseLossBackward0>)


In [133]:
# 비선형 모델 생성 (선형 모델 → 활성화 함수 → 선형 모델)

class Reg2(nn.Module):
    def __init__ (self, _dim):
        super(Reg2, self).__init__()
        # 다중 퍼셉트론 안에 선형 모델 → 활성화 함수 → 선형 모델
        self.model = nn.Sequential(
            # 첫번째 레이어
            nn.Linear(_dim, _dim),
            # 비선형 구조 파악용 활성화 함수
            nn.ReLU(),
            nn.Linear(_dim, 1)
        )
    
    def forward(self, x):
        return self.model(x)

In [134]:
model3 = Reg2(n_feature)
criterion3 = nn.MSELoss()
optimizer3 = optim.SGD(model3.parameters(), lr = 0.01)

In [135]:
# 반복 학습
for epoch in range(300):
    n = epoch + 1
    pred3 = model3(X_train_sc)
    loss3 = criterion3(pred3, y_train_tensor)
    optimizer3.zero_grad()
    loss3.backward()
    optimizer3.step()

    if n % 30 == 0:
        print(f"Epoch [{n} / 300], Loss: {round(loss3.item(), 6)}")

Epoch [30 / 300], Loss: 1.619349
Epoch [60 / 300], Loss: 0.87128
Epoch [90 / 300], Loss: 0.773977
Epoch [120 / 300], Loss: 0.74782
Epoch [150 / 300], Loss: 0.732087
Epoch [180 / 300], Loss: 0.719612
Epoch [210 / 300], Loss: 0.708609
Epoch [240 / 300], Loss: 0.698478
Epoch [270 / 300], Loss: 0.688843
Epoch [300 / 300], Loss: 0.679623


In [136]:
model3.eval()
with torch.no_grad():       # 버릇처럼 사용하기를 권장 (메모리 최적화)
    pred3 = model3(X_test_sc)
    loss3 = criterion3(pred3, y_test_tensor)

for i in range(10):
    print(f"실제 데이터: {y_test[i]}, 예측 데이터: {pred3[i].item()}")

실제 데이터: [0.477], 예측 데이터: 1.1895853281021118
실제 데이터: [0.458], 예측 데이터: 1.3360116481781006
실제 데이터: [5.00001], 예측 데이터: 2.2409262657165527
실제 데이터: [2.186], 예측 데이터: 2.5239195823669434
실제 데이터: [2.78], 예측 데이터: 1.9354417324066162
실제 데이터: [1.587], 예측 데이터: 2.0462563037872314
실제 데이터: [1.982], 예측 데이터: 2.465625286102295
실제 데이터: [1.575], 예측 데이터: 2.0323753356933594
실제 데이터: [3.4], 예측 데이터: 1.9483650922775269
실제 데이터: [4.466], 예측 데이터: 4.374147891998291


In [137]:
# 파이토치 랜덤 고정

torch.manual_seed(42)

In [140]:
# 딥러닝 분류 모델

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
df = pd.read_csv('../csv/iris.csv')

In [141]:
# 선형 모델을 이용하여 분류 → 로지스틱 회귀랑 비슷한 방식
# 출력값이 3개의 feature로 출력

X = df.drop('species', axis = 1)
y = df['species']

In [142]:
le = LabelEncoder()
y = le.fit_transform(y)
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [143]:
# train, test 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

In [144]:
# Scaler 작업

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [145]:
# Tensor 형태로 변환

X_train_tensor = torch.tensor(X_train_sc, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_sc, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.long)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)

In [146]:
# 모델 정의

class clf(nn.Module):
    def __init__(self, _dim):
        super(clf, self).__init__()
        self.model = nn.Linear(_dim, 3)
    
    def forward(self, x):
        return self.model(x)

In [147]:
clf_model = clf(X.shape[1])

In [149]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(clf_model.parameters(), lr = 0.01)

In [ ]:
pred = clf_model(X_train_tensor)
pred

In [151]:
torch.max(pred, 1)

torch.return_types.max(
values=tensor([0.1618, 0.4369, 1.2277, 0.0891, 0.5800, 1.3218, 0.5178, 1.0508, 1.9688,
        2.3230, 2.0149, 0.6627, 0.5671, 0.5614, 0.6435, 0.1627, 0.3329, 1.7175,
        0.9928, 0.2075, 1.2479, 0.2658, 2.3133, 0.0783, 0.5512, 1.1291, 1.2842,
        0.2460, 0.6946, 0.2714, 0.3145, 0.6495, 0.6456, 1.2248, 1.3334, 1.4135,
        0.4428, 0.2455, 1.6499, 0.8283, 0.2873, 1.0952, 0.6366, 1.9259, 0.1190,
        0.2445, 0.9191, 0.1787, 0.0657, 0.0934, 0.1708, 0.2055, 1.4473, 0.5623,
        0.2058, 0.7130, 0.4636, 1.5064, 0.2034, 0.9191, 0.1896, 0.7443, 1.0183,
        0.1533, 0.9874, 0.5843, 2.2442, 0.3701, 1.0327, 1.3257, 0.1694, 0.2351,
        0.2415, 1.3610, 0.5367, 1.5454, 0.5882, 1.2834, 1.1206, 0.9588, 0.1278,
        0.8986, 1.3429, 0.3459, 1.5273, 0.5364, 2.1302, 0.1067, 0.9258, 0.1270,
        0.6940, 1.2032, 0.3668, 0.2003, 1.4080, 2.4947, 1.3224, 1.2641, 0.9611,
        0.6100, 1.4683, 0.4062, 1.4551, 1.5790, 0.2708, 0.7993, 0.8099, 0.3117,
        0

In [155]:
clf_model.train()
for epoch in range(300):
    pred = clf_model(X_train_tensor)
    loss = criterion(pred, y_train_tensor)
    optimizer.zero_grad()       # 기울기 초기화
    loss.backward()             # 자동 미분
    optimizer.step()            # 기울기 업데이트
    n = epoch + 1
    
    if n % 30 == 0:
        print(f'Epoch: [{n} / 300], Loss: {round(loss.item(), 6)}')

Epoch: [30 / 300], Loss: 0.464252
Epoch: [60 / 300], Loss: 0.451952
Epoch: [90 / 300], Loss: 0.440959
Epoch: [120 / 300], Loss: 0.431048
Epoch: [150 / 300], Loss: 0.422042
Epoch: [180 / 300], Loss: 0.413802
Epoch: [210 / 300], Loss: 0.406215
Epoch: [240 / 300], Loss: 0.399191
Epoch: [270 / 300], Loss: 0.392656
Epoch: [300 / 300], Loss: 0.386548


In [156]:
# 평가
clf_model.eval()
with torch.no_grad():
    pred = clf_model(X_test_tensor)
    _, pred_idx = torch.max(pred, 1)

acc = accuracy_score(y_test, pred_idx)
print('정확도: ', round(acc, 4))
print(classification_report(y_test, pred_idx))

정확도:  0.8
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      0.40      0.57        10
           2       0.62      1.00      0.77        10

    accuracy                           0.80        30
   macro avg       0.88      0.80      0.78        30
weighted avg       0.88      0.80      0.78        30

